In [1]:
#1
# ============================================================
# Count max / average Qwen tokenizer tokens for:
# 1) relations[*]["relation"]
# 2) facts[*]["info"]
# in a JSON file on Google Drive
# ============================================================

!pip -q install transformers pandas tqdm

from google.colab import drive
drive.mount("/content/drive")

import json
import pandas as pd
from tqdm.auto import tqdm
from transformers import AutoTokenizer

Mounted at /content/drive


In [2]:
#2
# -----------------------------
# Config
# -----------------------------
MODEL_NAME = "Qwen/Qwen3-Embedding-8B"

JSON_PATH = "/content/drive/MyDrive/final_project/idea_1/kg/2wikimultihopqa/2wikimultihopqa_kg_extractions_all_00000001_to_00012685.clean.json"

ADD_SPECIAL_TOKENS = True

BATCH_SIZE = 1024

THRESHOLDS = [128, 256, 512, 1024]


# -----------------------------
# Load tokenizer only, not model
# -----------------------------
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
)

print("Loaded tokenizer:", MODEL_NAME)
print("Tokenizer model max length:", tokenizer.model_max_length)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/729 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.26k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Loaded tokenizer: Qwen/Qwen3-Embedding-8B
Tokenizer model max length: 131072


In [3]:
#3
# -----------------------------
# Load JSON
# -----------------------------
with open(JSON_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

print("Number of chunks:", len(data))


# -----------------------------
# Extract texts
# -----------------------------
relation_rows = []
fact_info_rows = []

for obj_i, item in enumerate(tqdm(data, desc="Extracting relation/info texts")):
    chunk_id = item.get("chunk_id")
    title = item.get("title")
    input_index = item.get("input_index")

    for rel_i, rel in enumerate(item.get("relations") or []):
        text = rel.get("relation")
        if text is not None:
            relation_rows.append({
                "section": "relations.relation",
                "chunk_id": chunk_id,
                "title": title,
                "input_index": input_index,
                "local_index": rel_i,
                "text": str(text),
                "head": rel.get("head"),
                "tail": rel.get("tail"),
            })

    for fact_i, fact in enumerate(item.get("facts") or []):
        text = fact.get("info")
        if text is not None:
            fact_info_rows.append({
                "section": "facts.info",
                "chunk_id": chunk_id,
                "title": title,
                "input_index": input_index,
                "local_index": fact_i,
                "text": str(text),
                "entity": fact.get("entity"),
            })

print("Number of relation texts:", len(relation_rows))
print("Number of fact info texts:", len(fact_info_rows))


# -----------------------------
# Token counting function
# -----------------------------
def count_tokens_for_rows(rows, batch_size=1024):
    counts = []

    for start in tqdm(range(0, len(rows), batch_size), desc="Counting tokens"):
        batch_rows = rows[start:start + batch_size]
        texts = [r["text"] for r in batch_rows]

        encoded = tokenizer(
            texts,
            add_special_tokens=ADD_SPECIAL_TOKENS,
            padding=False,
            truncation=False,
            return_attention_mask=False,
        )

        batch_counts = [len(ids) for ids in encoded["input_ids"]]
        counts.extend(batch_counts)

    for row, count in zip(rows, counts):
        row["token_count_qwen"] = count

    return rows


relation_rows = count_tokens_for_rows(relation_rows, BATCH_SIZE)
fact_info_rows = count_tokens_for_rows(fact_info_rows, BATCH_SIZE)


# -----------------------------
# Convert to DataFrames
# -----------------------------
df_rel = pd.DataFrame(relation_rows)
df_fact = pd.DataFrame(fact_info_rows)

df_all = pd.concat([df_rel, df_fact], ignore_index=True)


# -----------------------------
# Summary stats
# -----------------------------
def make_summary(df, section_name):
    s = df["token_count_qwen"]

    row = {
        "section": section_name,
        "count": int(s.count()),
        "min": int(s.min()) if len(s) else None,
        "max": int(s.max()) if len(s) else None,
        "mean": float(s.mean()) if len(s) else None,
        "median": float(s.median()) if len(s) else None,
        "p90": float(s.quantile(0.90)) if len(s) else None,
        "p95": float(s.quantile(0.95)) if len(s) else None,
        "p99": float(s.quantile(0.99)) if len(s) else None,
    }

    for t in THRESHOLDS:
        row[f"count_>{t}"] = int((s > t).sum())
        row[f"percent_>{t}"] = float((s > t).mean() * 100) if len(s) else None

    return row


summary_df = pd.DataFrame([
    make_summary(df_rel, "relations.relation"),
    make_summary(df_fact, "facts.info"),
])

print("\n========== SUMMARY ==========")
display(summary_df)


# -----------------------------
# Top longest examples
# -----------------------------
TOP_K = 20

top_rel = (
    df_rel
    .sort_values("token_count_qwen", ascending=False)
    .head(TOP_K)
    [["token_count_qwen", "chunk_id", "title", "input_index", "local_index", "text", "head", "tail"]]
)

top_fact = (
    df_fact
    .sort_values("token_count_qwen", ascending=False)
    .head(TOP_K)
    [["token_count_qwen", "chunk_id", "title", "input_index", "local_index", "text", "entity"]]
)

print("\n========== TOP LONGEST relations.relation ==========")
display(top_rel)

print("\n========== TOP LONGEST facts.info ==========")
display(top_fact)


Number of chunks: 12685


Extracting relation/info texts:   0%|          | 0/12685 [00:00<?, ?it/s]

Number of relation texts: 247252
Number of fact info texts: 184001


Counting tokens:   0%|          | 0/242 [00:00<?, ?it/s]

Counting tokens:   0%|          | 0/180 [00:00<?, ?it/s]

...

,section,count,min,max,mean,median,p90,p95,p99,count_>128,percent_>128,count_>256,percent_>256,count_>512,percent_>512,count_>1024,percent_>1024
0,relations.relation,247252,5,83,17.230773,16.0,25.0,28.0,35.0,0,0.000000,0,0.0,0,0.0,0,0.0
1,facts.info,184001,6,149,23.028239,22.0,32.0,36.0,44.0,1,0.000543,0,0.0,0,0.0,0,0.0



========== TOP LONGEST relations.relation ==========


,token_count_qwen,chunk_id,title,input_index,local_index,text,head,tail
14472,83,2wikimultihopqa_chunk_00000796,Andrés Manuel del Río,795,23,"Elementos de Orictognesia, o del conocimiento ...","Elementos de Orictognesia, o del conocimiento ...",Andrés Manuel del Río
14471,80,2wikimultihopqa_chunk_00000796,Andrés Manuel del Río,795,22,"Elementos de Orictognesia, o del conocimiento ...","Elementos de Orictognesia, o del conocimiento ...",1832
14470,76,2wikimultihopqa_chunk_00000796,Andrés Manuel del Río,795,21,"Elementos de Orictognesia, o del conocimiento ...","Elementos de Orictognesia, o del conocimiento ...",Philadelphia
98639,76,2wikimultihopqa_chunk_00005233,Prince August Leopold of Saxe-Coburg and Gotha,5232,13,Intercâmbio Cultural e Artístico nas relações ...,Intercâmbio Cultural e Artístico nas relações ...,Instituto Iberoamericano da Universidade Sofia
13358,75,2wikimultihopqa_chunk_00000736,Catholic University of Leuven (1834–1968),735,9,Discussion de la loi sur l'enseignement supéri...,Discussion de la loi sur l'enseignement supéri...,Th. Lesigne
98638,73,2wikimultihopqa_chunk_00005233,Prince August Leopold of Saxe-Coburg and Gotha,5232,12,Intercâmbio Cultural e Artístico nas relações ...,Intercâmbio Cultural e Artístico nas relações ...,Revista Iberoamericana
98637,71,2wikimultihopqa_chunk_00005233,Prince August Leopold of Saxe-Coburg and Gotha,5232,11,"Vinholes, L.C. wrote Intercâmbio Cultural e Ar...","Vinholes, L.C.",Intercâmbio Cultural e Artístico nas relações ...
98641,71,2wikimultihopqa_chunk_00005233,Prince August Leopold of Saxe-Coburg and Gotha,5232,15,Intercâmbio Cultural e Artístico nas relações ...,Intercâmbio Cultural e Artístico nas relações ...,1995
98640,70,2wikimultihopqa_chunk_00005233,Prince August Leopold of Saxe-Coburg and Gotha,5232,14,Intercâmbio Cultural e Artístico nas relações ...,Intercâmbio Cultural e Artístico nas relações ...,Tóquio
13359,70,2wikimultihopqa_chunk_00000736,Catholic University of Leuven (1834–1968),735,10,Discussion de la loi sur l'enseignement supéri...,Discussion de la loi sur l'enseignement supéri...,1844



========== TOP LONGEST facts.info ==========


,token_count_qwen,chunk_id,title,input_index,local_index,text,entity
25357,149,2wikimultihopqa_chunk_00001857,L. D. Bell High School,1856,13,The women's gymnastics team at L. D. Bell High...,Women's Gymnastics State Championships
35990,98,2wikimultihopqa_chunk_00002631,St Helens R.F.C.,2630,4,St Helens R.F.C. won the RFL Lancashire League...,St Helens R.F.C.
101662,91,2wikimultihopqa_chunk_00007083,Zabak,7082,3,Lata Mangeshkar gave playback for the actors i...,Lata Mangeshkar
131827,90,2wikimultihopqa_chunk_00009112,Heinz Rühmann,9111,9,Heinz Rühmann received a total of twelve Bambi...,Heinz Rühmann
135199,87,2wikimultihopqa_chunk_00009337,"Juan de la Cerda, 4th Duke of Medinaceli",9336,0,"Juan de la Cerda, 4th Duke of Medinaceli was t...","Juan de la Cerda, 4th Duke of Medinaceli"
73970,84,2wikimultihopqa_chunk_00005233,Prince August Leopold of Saxe-Coburg and Gotha,5232,7,Intercâmbio Cultural e Artístico nas relações ...,Intercâmbio Cultural e Artístico nas relações ...
73971,81,2wikimultihopqa_chunk_00005233,Prince August Leopold of Saxe-Coburg and Gotha,5232,8,Instituto Iberoamericano da Universidade Sofia...,Instituto Iberoamericano da Universidade Sofia
244,80,2wikimultihopqa_chunk_00000016,East High School (Denver),15,1,"In the 2018-2019 School Year, the racial demog...",East High School (Denver)
77707,80,2wikimultihopqa_chunk_00005490,"Princess Anne, Duchess of Calabria",5489,22,Madrid is the birthplace of Princess Cristina ...,Madrid
124310,78,2wikimultihopqa_chunk_00008605,Fred Negro,8604,1,"Turkeyneck Records released Drunks, Guns And L...",Turkeyneck Records
